# 常见的transforms


## pytorch中的`__call__`的用法


In [1]:
class Person:
    def __call__(self, name):
        print("__call__" + " Hello " + name)

    def hello(self, name):
        print("Hello" + name)

person = Person()
person("lizi")
person.hello("lizi")

__call__ Hello lizi
Hellolizi


前后有两条下划线的都是内置函数，一般不从外部直接用名字调用

可以看到call的作用


## transforms.ToTensor()

就是把一个 PIL Image 或 numpy.ndarray 转换成一个 tensor


## transforms.Normalize()的使用

Normalize 是逐通道的标准化，公式：`output = (input - mean) / std`。作用是把数据分布拉到 0 均值、单位方差附近，让训练更稳、收敛更快。

### 函数签名

```python
transforms.Normalize(mean, std, inplace=False)
```

### 参数说明

- **mean**：每个通道的均值，写成序列。三通道图片就写三个数，例如 `[0.485, 0.456, 0.406]`
- **std**：每个通道的标准差，例如 `[0.229, 0.224, 0.225]`
- **inplace**：是否原地修改，默认 False。不要开，开了会破坏原来的数据

常用的 ImageNet 统计量：`mean=[0.485, 0.456, 0.406]`，`std=[0.229, 0.224, 0.225]`。这是 ImageNet 上统计出来的 RGB 三通道均值和标准差，用预训练模型时按这套值做预处理。

### 值域变化（实测 0013035.jpg）

| 处理 | 值域 |
| --- | --- |
| ToTensor 之后 | 0.0 ~ 1.0 |
| Normalize(mean=[0.5]*3, std=[0.5]*3) 之后 | -1.0 ~ 1.0 |
| Normalize 用 ImageNet 统计量之后 | -2.1179 ~ 2.64 |
| 反归一化 t * 0.5 + 0.5 | 0.0 ~ 1.0 |

### 三个注意点

1. **必须写在 ToTensor 后面**。Normalize 只吃张量，直接传 PIL 图片会报 `TypeError: img should be Tensor Image. Got <class 'PIL.JpegImagePlugin.JpegImageFile'>`
2. **mean/std 的个数要和通道数对得上**。写 1 个会自动广播到所有通道；写 2 个或 4 个会报 `RuntimeError: The size of tensor a (3) must match the size of tensor b (2)`
3. **归一化后不能直接 add_image**。值域不再是 0~1、有负数，TensorBoard 会把负值截成 0，图发灰发黑，要先反归一化

### 反归一化

```python
mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
img_show = img_norm * std + mean
```

### 顺序不能错

```python
transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),        # 先转张量，值域 0~1
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),   # 再标准化
])
```

预处理要训练和推理保持一致：训练时用了 Normalize，推理（test.py）也必须用同一个，否则准确率会掉。

### 示例

```python
from PIL import Image
import torch
from torchvision import transforms

img_path = '/Users/daxian/deep-learning/dataset/train/ants_image/0013035.jpg'
img = Image.open(img_path)

trans = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

img_norm = trans(img)
print(img_norm.shape, img_norm.dtype)                 # torch.Size([3, 224, 224]) torch.float32
print(img_norm.min().item(), img_norm.max().item())   # 有负数

# 反归一化，变回 0~1 才能送进 add_image 查看
mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
img_show = img_norm * std + mean
print(img_show.min().item(), img_show.max().item())
```
